In [17]:
import pandas as pd

csv_df=pd.read_csv("GIS Dashboard.csv")

csv_df.columns = [
    'constituency', 'project_id', 'project_name', 'state',
    'districts', 'nh_number', 'length_km', 'lane_config',
    'work_type', 'contract_type', 'sanctioned_amt_cr',
    'physical_progress_pct', 'start_date', 'end_date',
    'piu_city', 'piu_contact', 'ro_name', 'ro_contact'
]

In [19]:
print(csv_df['piu_city'].unique())
print(csv_df['piu_city'].nunique())

<StringArray>
[    'Kamareddy',    'Mancherial',     'Ahmedabad',   'Ahilyanagar',
         'Ajmer',        'Jaipur',        'Kanpur',        'Jhansi',
 'Amravati (MH)',     'Cochin-II',
 ...
        'Raipur',      'Shillong',        'Shimla',     'Karaikudi',
    'Sonipat-DL',           'Goa',     'Tuticorin',    'Chennai-II',
        'Ramban',       'Vidisha']
Length: 190, dtype: str
190


In [24]:
print(csv_df['piu_city'].unique())

<StringArray>
[    'Kamareddy',    'Mancherial',     'Ahmedabad',   'Ahilyanagar',
         'Ajmer',        'Jaipur',        'Kanpur',        'Jhansi',
 'Amravati (MH)',     'Cochin-II',
 ...
        'Raipur',      'Shillong',        'Shimla',     'Karaikudi',
    'Sonipat-DL',           'Goa',     'Tuticorin',    'Chennai-II',
        'Ramban',       'Vidisha']
Length: 190, dtype: str


In [25]:
print(csv_df['nh_number'].value_counts())

nh_number
44        66
27        50
48        41
53        37
16        37
          ..
722        1
86         1
146346     1
346        1
NH 361     1
Name: count, Length: 376, dtype: int64


In [27]:
mask = ~csv_df['nh_number'].astype(str).str.match(r'^\d+$')
print(csv_df[mask]['nh_number'].value_counts())
print(csv_df[mask].shape[0], "total dirty rows")

nh_number
TBD             28
754K            27
148N            25
544G            14
NE-4            14
                ..
1A               1
766E & 766EE     1
4A               1
139W             1
NH 361           1
Name: count, Length: 200, dtype: int64
467 total dirty rows


In [28]:
print(csv_df.shape[0])

1351


In [29]:
csv_df['nh_number_clean'] = (
    csv_df['nh_number']
    .astype(str)
    .str.replace(r'^NH\s+', '', regex=True)  # remove "NH " prefix
    .str.strip()
)

# mark TBD as NaN
csv_df.loc[csv_df['nh_number_clean'] == 'TBD', 'nh_number_clean'] = None

print(csv_df['nh_number_clean'].isna().sum(), "rows with no NH number")

30 rows with no NH number


In [31]:
csv_df['nh_number_clean']

0          44
1         363
2          44
3         363
4          08
        ...  
1346    248BB
1347      161
1348      161
1349      161
1350       44
Name: nh_number_clean, Length: 1351, dtype: str

In [34]:
csv_df['nh_number_clean'].value_counts().tail(50)

nh_number_clean
332                 1
8B                  1
24B                 1
547E                1
353J                1
353D                1
75 & 23             1
320B                1
NH-33               1
NH-71               1
344 N               1
146                 1
934                 1
42                  1
7 ,SH-10            1
135BG               1
227 F               1
05                  1
555                 1
21A                 1
9/10                1
38, 383             1
536                 1
150E                1
709 ,352A           1
NE-II               1
344P                1
566                 1
117                 1
NE-7 & NH-48        1
NH-32               1
07,34               1
183                 1
83, 336, 36, 536    1
944                 1
PC                  1
NH-5                1
N H-32              1
552                 1
244                 1
1A                  1
169                 1
766E & 766EE        1
748                 1
4A              

In [49]:
null_values = [
    'TBD', 'PC', '146346', 'nan', 'Greenfield',
    'NE 4', '65(New NH-52', 'Yet to be decided',
    'Old NH-24(New NH-30)', '30 (Old NH-7)',
    '27E.W.', '8A (Extn.)', 'RING ROAD', 'Outer Ring Road',
    '65(New NH-52', '30 (Old NH-7)', '27E.W.'
]

def clean_nh(val):
    val = str(val).strip()
    if val in null_values:
        return None
    import re
    val = re.sub(r'^N\s*H[-\s]*', '', val, flags=re.IGNORECASE).strip()
    val = re.split(r'\s*[,&/]\s*|\s+and\s+', val, flags=re.IGNORECASE)[0].strip()
    val = re.sub(r'(\d+)\s+([A-Za-z])', r'\1\2', val).strip()
    return val if val else None

csv_df['nh_number_clean'] = csv_df['nh_number'].apply(clean_nh)
csv_df = csv_df[csv_df['nh_number_clean'].notna()].reset_index(drop=True)
print(csv_df.shape[0], "clean rows remaining")

1304 clean rows remaining


In [51]:
# anything still obviously wrong?
weird = csv_df[csv_df['nh_number_clean'].str.contains(r'and|,|&|/', na=False)]
print(weird['nh_number_clean'].value_counts())

Series([], Name: count, dtype: int64)


In [ ]:
# after running clean_nh
csv_df.loc[csv_df['nh_number_clean'].str.contains(r'[\(\)\.]', na=False), 'nh_number_clean'] = None

In [54]:
import re
weird = csv_df[csv_df['nh_number_clean'].str.match(r'^[A-Za-z0-9\-]+$') == False]
print(weird['nh_number_clean'].unique())

<StringArray>
[nan]
Length: 1, dtype: str
